In [ ]:
from encoding.assembly.assembly_generator import AssemblyGenerator
from encoding.assembly.assembly_loader import load_assembly
import os
import glob
import numpy as np
from copy import deepcopy
from encoding.assembly.assemblies import SimpleNeuroidAssembly
from encoding.features.factory import FeatureExtractorFactory
from encoding.downsample.downsampling import Downsampler
from encoding.models.nested_cv import NestedCVModel
from encoding.trainer import AbstractTrainer

In [ ]:
#example for sub-UTS05
assembly = AssemblyGenerator.generate_assembly(
    dataset_type="lebel",
    data_dir="/path/to/litcoder_release/data/lebel/neural_data/",
    subject="UTS05",
    tr=2,
    lookback=256,
    context_type="fullcontext"
)

In [ ]:
stories_to_keep = ['adollshouse',
 'adventuresinsayingyes',
 'avatar',
 'buck',
 'eyespy',
 'fromboyhoodtofatherhood',
 'hangtime',
 'haveyoumethimyet',
 'inamoment',
 'itsabox',
 'legacy',
 'naked',
 'odetostepfather',
 'sloth',
 'souls',
 'stagefright',
 'swimmingwithastronauts',
 'thatthingonmyarm',
 'theclosetthatateeverything',
 'tildeath',
 'undertheinfluence',
 'wheretheressmoke']

filtered_story_data_list = [
    assembly.story_data[story] 
    for story in stories_to_keep
]

# Map story names to files
folder = "/path/to/ProjectedComponentsEncoding/sub-UTS05/"
files = glob.glob(os.path.join(folder, "*.npy"))
story_to_file = {}
for f in files:
    basename = os.path.basename(f)
    if "task-" in basename:
        task_part = basename.split("task-")[1]
        story_name = task_part.split("_")[0]
        story_to_file[story_name] = f

print(story_to_file)

# Update brain data 
updated_story_data_list = []
for story_data in filtered_story_data_list:
    new_story_data = deepcopy(story_data)
    
    story_name = story_data.name
    path = story_to_file[story_name]
    new_brain_data = np.load(path).T
    
    # Remove last 10 TRs
    #new_brain_data = new_brain_data[:-10, :]
    
    # Verify shapes match
    print(f"{story_name}: Old shape {story_data.brain_data.shape}, New shape {new_brain_data.shape}")
    
    if new_brain_data.shape[0] != story_data.brain_data.shape[0]:
        print(f"  WARNING: Timepoint mismatch for {story_name}!")
        print(f"   Expected {story_data.brain_data.shape[0]}, got {new_brain_data.shape[0]}")
    
    new_story_data.brain_data = new_brain_data
    updated_story_data_list.append(new_story_data)

#Create new assembly
new_assembly = SimpleNeuroidAssembly(
    story_data_list=updated_story_data_list,
    validation_method=assembly.validation_method
)

print(f"\nNew assembly shape: {new_assembly.shape}")
print(f"Stories included: {new_assembly.stories}")

In [ ]:
#for Pythia
fir_delays = [1, 2, 3, 4, 5]
trimming_config = {
    "train_features_start": 10, "train_features_end": -5,
    "train_targets_start": 0,  "train_targets_end": None,
    "test_features_start": 50,  "test_features_end": -5,
    "test_targets_start": 40,   "test_targets_end": None,
}
downsample_config = {
    "method": "lanczos", 
    "window": 3,          
    "cutoff_mult": 1.0,     
}

downsampler = Downsampler()
model = NestedCVModel(model_name="ridge_regression")

for layer_idx in range(24):  
    print(f"\n=== Running layer {layer_idx} ===")
    
    # Create extractor for this specific layer
    extractor = FeatureExtractorFactory.create_extractor(
        modality="language_model",
        model_name="EleutherAI/pythia-410m",
        config={
            "layer_idx": layer_idx,
            "last_token": True,
            "lookback": 256,
            "context_type": "fullcontext",
        },
        cache_dir="cache_language_model_Pythia",
    )
    
    run_name = f"Pythia_layer{layer_idx}"
    results_dir = os.path.join("FinalResults/sub-UTS05/Component/Pythia", run_name)
    os.makedirs(results_dir, exist_ok=True)
    
    # Initialize trainer
    trainer = AbstractTrainer(
        assembly=new_assembly,  
        feature_extractors=[extractor],
        downsampler=downsampler,
        model=model,
        trimming_config=trimming_config,
        fir_delays=fir_delays,
        use_train_test_split=True,
        layer_idx=layer_idx,  
        lookback=256,  
        logger_backend="wandb",
        wandb_project_name="lebel-Pythia-layers-5TRS",
        dataset_type="lebel",
        results_dir=results_dir,
        downsample_config=downsample_config,
        plot_results=False,
        run_name=run_name,
    )
    
    # Train and get metrics
    metrics = trainer.train()
    
    # Print summary for this layer
    print({
        "layer": layer_idx,
        "median_correlation": metrics.get("median_score", float("nan")),
        "n_significant": metrics.get("n_significant"),
    })
    
    print(f"Layer {layer_idx} complete. Results saved to {results_dir}")

In [ ]:
#for gpt
fir_delays = [1, 2, 3, 4, 5]
trimming_config = {
    "train_features_start": 10, "train_features_end": -5,
    "train_targets_start": 0,  "train_targets_end": None,
    "test_features_start": 50,  "test_features_end": -5,
    "test_targets_start": 40,   "test_targets_end": None,
}
downsample_config = {
    "method": "lanczos", 
    "window": 3,          
    "cutoff_mult": 1.0,     
}

downsampler = Downsampler()
model = NestedCVModel(model_name="ridge_regression")

for layer_idx in range(12):  
    print(f"\n=== Running layer {layer_idx} ===")
    
    # Create extractor for this specific layer
    extractor = FeatureExtractorFactory.create_extractor(
        modality="language_model",
        model_name="gpt2-small",
        config={
            "layer_idx": layer_idx,
            "last_token": True,
            "lookback": 256,
            "context_type": "fullcontext",
        },
        cache_dir="cache_language_model_subUTS01",
    )
    
    run_name = f"Pythia_layer{layer_idx}"
    results_dir = os.path.join("FinalResults/sub-UTS05/Component/GPT/", run_name)
    os.makedirs(results_dir, exist_ok=True)
    
    # Initialize trainer
    trainer = AbstractTrainer(
        assembly=new_assembly,  
        feature_extractors=[extractor],
        downsampler=downsampler,
        model=model,
        trimming_config=trimming_config,
        fir_delays=fir_delays,
        use_train_test_split=True,
        layer_idx=layer_idx,  
        lookback=256,  
        logger_backend="wandb",
        wandb_project_name="lebel-gpt-layers-5TRS",
        dataset_type="lebel",
        results_dir=results_dir,
        downsample_config=downsample_config,
        plot_results=False,
        run_name=run_name,
    )
    
    # Train and get metrics
    metrics = trainer.train()
    
    # Print summary for this layer
    print({
        "layer": layer_idx,
        "median_correlation": metrics.get("median_score", float("nan")),
        "n_significant": metrics.get("n_significant"),
    })
    
    print(f"Layer {layer_idx} complete. Results saved to {results_dir}")